### Bronze: `01_bronze.yellow_trips_raw` — trips exactly as delivered
No rows dropped, no values changed. Only `processing_timestamp` is added (audit column, ignored downstream).

In [0]:
%run ../../config/config

In [0]:
df = spark.read.format("parquet").load(TRIPS_FILE)
df.printSchema()

In [0]:
df.select("tpep_dropoff_datetime","tpep_pickup_datetime").show(5)

In [0]:
from pyspark.sql.functions import col, current_timestamp

df = spark.read.format("parquet").load(TRIPS_FILE)

# Timestamps are naive NY wall-clock values. With session zone = UTC this cast keeps them unchanged.
df = df.withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("timestamp")) \
       .withColumn("tpep_dropoff_datetime", col("tpep_dropoff_datetime").cast("timestamp")) \
       .withColumn("processing_timestamp", current_timestamp())

In [0]:
df.printSchema()

In [0]:
# df.select("processing_timestamp","tpep_dropoff_datetime","tpep_pickup_datetime").show(5)

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tbl("01_bronze", "yellow_trips_raw"))